## LexRank Extractive Summarizer

**Objective:** Implement a graph-based extractive summarizer using LexRank, and evaluate it using standard ROUGE metrics.

**Dataset:** Same 500-article WikiHow sample used in the TF-IDF notebook (loaded via saved CSV for consistency across notebooks).

**Steps:**
1. Load the pre-sampled and cleaned dataset
2. Parse each article's raw text using `sumy`'s `PlaintextParser`
3. Apply `LexRankSummarizer` to build a sentence similarity graph (cosine similarity on TF-IDF vectors) and rank sentences via eigenvector centrality
4. Select the top 5 ranked sentences as the summary
5. Evaluate against reference headlines using ROUGE-1, ROUGE-2, and ROUGE-L (F-measure)

**Evaluation:** The generated summaries are evaluated using `rouge-score` library.

**Results (mean across 500 articles):**

| Metric | Score |
|---|---|
| ROUGE-1 | 0.278 |
| ROUGE-2 | 0.078 |
| ROUGE-L | 0.152 |

LexRank improves modestly over TF-IDF across all ROUGE metrics by accounting for inter-sentence relationships rather than relying on word frequency alone.



In [ ]:
import sumy
import pandas as pd
import rouge_score

In [2]:
sample = pd.read_csv('sample.csv')

In [3]:
sample.head(3)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,['Meditating is a great way to relax your mind...,"When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"['The beauty industry is a huge one, and it’s ...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954
2,Relax. Make your move before you psyche yourse...,How to Ask for a Phone Number,If there's one single thing you can do to make...,2415,"[""If there's one single thing you can do to ma...",Though it's always difficult (some might say a...,{'rouge1': Score(precision=0.20652173913043478...,0.301587,0.058511,0.153439


In [4]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer


In [5]:
parser = PlaintextParser.from_string(sample['text'].iloc[0], Tokenizer("english"))


In [6]:
parser.document.sentences

(<Sentence: Meditating is a great way to relax your mind, and you can meditate almost anywhere and at any time.>,
 <Sentence: Just pick a quiet place where you can sit on level ground and close your eyes.>,
 <Sentence: Cross your legs and keep your hands on your lap.>,
 <Sentence: Focus on inhaling and exhaling, and let your body be governed by your breath.>,
 <Sentence: Keep as still as possible and avoid fidgeting.>,
 <Sentence: Be aware of what you can't control.>,
 <Sentence: Focus and absorb the smells and sounds around you.>,
 <Sentence: Clear your mind.>,
 <Sentence: Don't think about how much work you have left to do, or about what you're going to make for dinner.>,
 <Sentence: Just focus on clearing your mind and managing your breath.>,
 <Sentence: Relax every part of your body.>,
 <Sentence: You can focus on one part of your body at a time until you feel that every part of you is loose and relaxed.>,
 <Sentence: Going out to the movies or watching a movie on television can he

In [7]:
#Initalize LexRank Summarizer:
summarizer = LexRankSummarizer()

In [8]:
summary_sentences = summarizer(parser.document, 5)
# 5 is the number of sentences in the output summary, not the input.
# outputs the top 5 most important ones.
summary_sentences

(<Sentence: Going out to the movies or watching a movie on television can help you escape into another universe and to take your mind off of your own problems.>,
 <Sentence: When you watch a movie, try to clear your mind as much as possible and think about what the characters are doing and saying instead of what you're going to do or say after the movie.>,
 <Sentence: If you have a busy schedule, choose a board game night or go out to see a comedy with your friends, instead of going to a crowded bar where you won't have as much of a chance to laugh.>,
 <Sentence: If you love to drive, then going for a long drive late at night can help you relax and feel more in control of your life.>,
 <Sentence: Reading will not only improve your knowledge, but it will allow you to rest your body and quiet your mind as you focus on the material in front of you.>)

**NOTE:----** <br>  
**1.LexRank Tldr:----**
```python
PlaintextParser.from_string(sample['text'].iloc[0], Tokenizer("english"))
```

takes just the first  row text column value and tokenizes it directly using LexRank's Own Tokenizer and stores it in parser.document.
And after that those tokenized sentences are passed to lexrank summarizer - it takes all sentences in  parser.document as input,and then outputs the top 5 most important sentences as the summary of the whole text.  

-->The summarizer doesn't take only 5 sentences as input — it takes all sentences in  parser.document as input,and then outputs the top 5 most important ones.   

*Q.How does lexrank summarizer know which 5 sentences to choose for summary.*  [IMP]  
(It builds some kind of graph and then select the sentences (which are nodes of graph) which are most connected to other important sentences. This is done by computing eigenvector centrality score for each sentences i.e. node in the graph.)
                                                                                     

**2.TF-IDF Tldr:----**
1. In tfidf we need to separately tokenize the sentences and then feed that to tfidf summarizer.
2. tfidf summarizer generates scores for each words in every sentences.   
*On what basis tfidf genrates scores for each words in sentences? --- on the basis of how often they appear(TF) / IDF*
3. We sum those scores and get score for each sentence (stored in sentence_scores) and sort it and map it back to the top 5 tokenized sentences....which is the summary.            
But in LexRank, sentence tokenization is done by LexRank's own tokenizer.

In [9]:
# str() is a built-in function used to convert any object into its string representation.
summary = ' '.join(str(sentences) for sentences in summary_sentences)
summary

"Going out to the movies or watching a movie on television can help you escape into another universe and to take your mind off of your own problems. When you watch a movie, try to clear your mind as much as possible and think about what the characters are doing and saying instead of what you're going to do or say after the movie. If you have a busy schedule, choose a board game night or go out to see a comedy with your friends, instead of going to a crowded bar where you won't have as much of a chance to laugh. If you love to drive, then going for a long drive late at night can help you relax and feel more in control of your life. Reading will not only improve your knowledge, but it will allow you to rest your body and quiet your mind as you focus on the material in front of you."

### LexRank Summarizer Function--------

In [10]:
def lexrank_summarizer(text, num_sentences=5):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LexRankSummarizer()
    summary_sentences = summarizer(parser.document, num_sentences)
    summary = ' '.join(str(sentences) for sentences in summary_sentences)
    return summary

In [11]:
sample['lexrank_summary'] = sample['text'].apply(lexrank_summarizer)

In [12]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,['Meditating is a great way to relax your mind...,"When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"['The beauty industry is a huge one, and it’s ...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...


### **Evaluation ----**

In [13]:
from rouge_score import rouge_scorer

In [14]:
# Intialize the scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

In [15]:
scores = scorer.score(sample['headline'].iloc[0], sample['lexrank_summary'].iloc[0])
scores

{'rouge1': Score(precision=0.08074534161490683, recall=0.6842105263157895, fmeasure=0.14444444444444443),
 'rouge2': Score(precision=0.04375, recall=0.3888888888888889, fmeasure=0.07865168539325841),
 'rougeL': Score(precision=0.08074534161490683, recall=0.6842105263157895, fmeasure=0.14444444444444443)}

In [16]:
sample['lexrank_score'] = sample.apply(lambda x: scorer.score(x['headline'], x['lexrank_summary']), axis=1)

In [17]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,['Meditating is a great way to relax your mind...,"When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,"{'rouge1': (0.08074534161490683, 0.68421052631..."
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"['The beauty industry is a huge one, and it’s ...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,"{'rouge1': (0.17355371900826447, 0.36206896551..."


In [18]:
sample['lexrank_score'].iloc[0]['rouge1'].fmeasure

0.14444444444444443

In [19]:
sample['lexrank_rouge1_f'] = sample['lexrank_score'].apply(lambda x : x['rouge1'].fmeasure)
sample['lexrank_rouge2_f'] = sample['lexrank_score'].apply(lambda x : x['rouge2'].fmeasure)
sample['lexrank_rougeL_f'] = sample['lexrank_score'].apply(lambda x : x['rougeL'].fmeasure)

In [20]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,['Meditating is a great way to relax your mind...,"When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,"{'rouge1': (0.08074534161490683, 0.68421052631...",0.144444,0.078652,0.144444
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"['The beauty industry is a huge one, and it’s ...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,"{'rouge1': (0.17355371900826447, 0.36206896551...",0.234637,0.022599,0.134078


In [21]:
print(sample[['lexrank_rouge1_f', 'lexrank_rouge2_f', 'lexrank_rougeL_f']].mean())

lexrank_rouge1_f    0.313287
lexrank_rouge2_f    0.080646
lexrank_rougeL_f    0.170346
dtype: float64


### LexRank summarizer is complete. ✅


In [22]:
sample.to_csv('sample1.csv', index=False)